# 0416 · Comparación de modelos y decisión, versión 2 (OE3, OE4, H1, H2)

> **Versión 2 (05-11-2026, evaluación del director).** Solo validación: **no hay prueba final en este cuaderno**. La prueba independiente será septiembre y octubre (reserva que extrae 0110), y se corre una sola vez después de registrar y enviar `P4b_decision_v2.json`. Cambios:
> - **C2:** cinco bloques de validación (abril a agosto); julio y agosto dejan de ser prueba.
> - **C3:** referencias de mediana (zona×franja y cuantil 0,5) y búsqueda resumida por función de pérdida; H1 también contra la referencia del mismo tipo (L1 contra mediana, L2 contra media).
> - **A5:** en clasificación, referencias de tasa histórica zona×franja y «menor margen primero».
> - **C4:** alertas con capacidad fija de k guías por día (k = 250, 500 y 1.000); el modelo de clasificación se elige por exhaustividad con k = 500.
> - **M3:** H2 también para la logística (con y sin clima).
> - **A6:** comprueba la huella de la base antes de usarla.
> - Tabla por día de la semana (sábados) con la probabilidad media, solo en validación.

| Sección | Qué pasa |
| --- | --- |
| 1. Juntar resultados | Predicciones de validación de todos los modelos, importancias y búsqueda |
| 2. Resultados de validación | Tablas general, por bloque y por segmento |
| 3. H1 y H2 | Diferencias con IC 95 % por días |
| 4. Capacidad fija y modelo propuesto | Exhaustividad y precisión con k alertas por día; elección |
| 5. Calibración y día de la semana | Calibración cruzada; tabla de sábados |
| 6. Decisión | `P4b_decision_v2.json` (con `DECISION_APROBADA = True`) |
| 7. Manifiesto | Versiones, parámetros y salidas |

**Requisitos.** 0411 v2 y los cuadernos de modelos v2: 0412 (referencias y lineal), 0413 (Random Forest), 0414 (XGBoost) y 0415 (LightGBM).

**Salidas.** `P4b_*` en `REVISION_2026_11/V4/`.

In [ ]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [ ]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

## 0. Configuración

In [ ]:
import os, json, glob, sys, time, datetime, hashlib, warnings
import numpy as np, pandas as pd, duckdb, joblib
import lightgbm as lgb, xgboost as xgb, sklearn
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (mean_absolute_error, average_precision_score, precision_score, recall_score, f1_score)
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas del espacio de trabajo

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (
    cargar_preparacion, verificar_huella, ordenar, mejores, params_de, PERDIDA_CORTA,
    metricas, tabla_metricas, metricas_por_segmento, diferencias, diferencia_tarea, diferencia_capacidad_por_bloques,
    calibracion_cruzada, calibracion_tabla, tabla_umbrales, umbral_por_capacidad, tabla_capacidad, alertas_por_capacidad, huella)

In [ ]:
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"   # lo que dejaron 0411 a 0415 v2
CARPETA_MODELOS  = f"{CARPETA_SALIDAS}MODELOS/"    # calibrador de validación (si se decide calibrar)
os.makedirs(CARPETA_MODELOS, exist_ok=True)

ALERTAS_POR_DIA = 500                 # C4: capacidad principal (escenario, no dato oficial de la empresa)
CAPACIDADES     = [250, 500, 1000]    # C4: sensibilidad de la capacidad
B_BOOTSTRAP     = 1000                # remuestreos por días para los IC
MODELOS   = ["Lineal", "RandomForest", "XGBoost", "LightGBM"]   # "Lineal" = regresión lineal o logística según la tarea
ENSAMBLES = ["RandomForest", "XGBoost", "LightGBM"]
REFERENCIAS = {"clf": ["Referencia", "Referencia_tasa_historica", "Referencia_menor_margen"],
               "reg": ["Referencia", "Referencia_mediana", "Lineal_mediana"]}

In [ ]:
# Configuración común que fijó 0411 v2 (A6: primero se comprueba que la base no cambió)
HUELLA_BASE = verificar_huella(CARPETA_MODELADO)
# Memoria: aquí no se necesita la base completa ni la historia; solo la configuración y la columna de guía.
CFG = json.load(open(f"{CARPETA_MODELADO}configuracion.json")); CFG["TAREAS"] = [tuple(t) for t in CFG["TAREAS"]]
_guias = pd.read_parquet(f"{CARPETA_MODELADO}base_modelado.parquet", columns=["guia"]).guia
COD_GUIA = pd.Series(np.arange(len(_guias), dtype="int32"), index=_guias.values); del _guias
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
INICIOS_VALIDACION, FIN_DESARROLLO = CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"]
CORTE_OBSERVACION, FIN_EVALUABLE, P99_T0_A_ENTREGA_H = CFG["CORTE_OBSERVACION"], CFG["FIN_EVALUABLE"], CFG["P99_T0_A_ENTREGA_H"]
USAR_VARIABLES_CLIENTE, SOLO_SLA_UNICO = CFG["USAR_VARIABLES_CLIENTE"], CFG["SOLO_SLA_UNICO"]
EXCLUIR_ENTREGA_SIN_HORA = CFG["EXCLUIR_ENTREGA_SIN_HORA"]
ruta, RUTA_CALENDARIO = CFG["ruta_gold"], CFG["RUTA_CALENDARIO"]
exclusiones = pd.DataFrame(CFG["exclusiones"])   # misma tabla que P4b_exclusiones.csv
print(f"{len(COD_GUIA):,} guías · versiones: {list(VERSIONES)} · bloques: {INICIOS_VALIDACION} · huella {HUELLA_BASE[:16]}...")

## 1. Juntar los resultados de cada modelo

### 1.1 ¿Están todos?

In [ ]:
esperados = {"0412": ["pred_val_Referencia.parquet", "pred_val_Lineal.parquet"],
             "0413": ["pred_val_RandomForest.parquet", "importancia_RandomForest.parquet", "parametros_RandomForest.json"],
             "0414": ["pred_val_XGBoost.parquet", "importancia_XGBoost.parquet", "parametros_XGBoost.json"],
             "0415": ["pred_val_LightGBM.parquet", "importancia_LightGBM.parquet", "parametros_LightGBM.json"]}
faltan = {c: [f for f in fs if not os.path.exists(CARPETA_MODELADO + f)] for c, fs in esperados.items()}
faltan = {c: fs for c, fs in faltan.items() if fs}
assert not faltan, f"Faltan resultados; correr primero estos cuadernos: {faltan}"
print("Están los resultados de los cinco modelos.")

### 1.2 Predicciones de validación

Una tabla con las predicciones de todos los modelos, en el mismo orden que tendría si todo se hubiera corrido en un solo cuaderno.

In [ ]:
archivos = [f"{CARPETA_MODELADO}pred_val_{m}.parquet" for m in ["Referencia"] + MODELOS] + \
           sorted(glob.glob(f"{CARPETA_MODELADO}pred_val_*_por_perdida.parquet"))   # C3 (0414 y 0415 v2)
MODELOS_H2 = ["LightGBM", "Lineal"]   # únicos modelos que se comparan con y sin clima (M3)
def cargar_pred(f):
    """Predicciones de validación con tipos compactos (la tabla completa tiene decenas de millones de filas):
    guía como entero, día como número de días desde 1970, texto repetido compartido y números en float32."""
    p = pd.read_parquet(f)
    p = p[(p.version == "internas") | p.modelo.isin(MODELOS_H2)]   # con clima solo hace falta lo que entra en H2
    p["guia"] = p.guia.map(COD_GUIA).astype("int32")
    p["dia"] = pd.to_datetime(p.dia).values.astype("datetime64[D]").astype("int64").astype("int32")
    p["bloque"] = p.bloque.astype(str)
    for c in ["version", "tarea", "modelo", "bloque", "categoria_zona", "franja"]:
        p[c] = p[c].astype("category").astype(object)
    p["y"], p["pred"] = p.y.astype("float32"), p.pred.astype("float32")
    return p
import gc
pred = []
for f in archivos:
    pred.append(cargar_pred(f)); gc.collect()
pred = pd.concat(pred, ignore_index=True); gc.collect()
print(f"{len(pred):,} filas de predicciones · {pred.memory_usage(deep=False).sum() / 1e9:.1f} GB")
ORDEN_MODELOS = REFERENCIAS["clf"] + REFERENCIAS["reg"][1:] + MODELOS + sorted(set(pred.modelo) - set(REFERENCIAS["clf"] + REFERENCIAS["reg"] + MODELOS))
ORDEN_MODELOS = list(dict.fromkeys(m for m in ORDEN_MODELOS if m in set(pred.modelo)))
# (Se quitó el reordenamiento y la copia completa a P4b_predicciones_validacion.parquet: duplicaban la memoria.
#  Las predicciones de cada modelo siguen en MODELADO/pred_val_*.parquet.)
print(pred.groupby(["version", "tarea", "modelo"]).size().unstack(level=[0, 1]))

### 1.3 Importancia por permutación de cada bloque (insumo de 0420)

In [ ]:
importancias_pliegues = pd.concat([pd.read_parquet(f"{CARPETA_MODELADO}importancia_{m}.parquet") for m in ENSAMBLES], ignore_index=True)
importancias_pliegues = ordenar(importancias_pliegues, {"tarea": ["clf", "reg"], "bloque": None, "modelo": ENSAMBLES})
importancias_pliegues.to_csv(f"{CARPETA_SALIDAS}P4b_importancia_pliegues.csv", index=False)

### 1.4 Hiperparámetros elegidos en cada ensamble

In [ ]:
PARAMETROS = {m: json.load(open(f"{CARPETA_MODELADO}parametros_{m}.json")) for m in ENSAMBLES}
PARAMS_BASE = {**{m: PARAMETROS[m]["params_base"] for m in ENSAMBLES}, "Lineal": {}}
ESPACIOS = {m: PARAMETROS[m]["espacio"] for m in ["LightGBM", "XGBoost", "RandomForest"]}
n_configs = {m: PARAMETROS[m]["n_configs"] for m in ENSAMBLES}   # un solo número si todos usaron el mismo
max_filas = {m: PARAMETROS[m]["max_filas"] for m in ENSAMBLES}
N_CONFIGS_BUSQUEDA = n_configs["LightGBM"] if len(set(n_configs.values())) == 1 else n_configs
MAX_FILAS_BUSQUEDA = max_filas["LightGBM"] if len(set(max_filas.values())) == 1 else max_filas

In [ ]:
busquedas = [f"{CARPETA_MODELADO}busqueda_{m}.pkl" for m in ENSAMBLES]
if all(os.path.exists(f) for f in busquedas):
    tabla_busqueda = ordenar(pd.concat([pd.read_pickle(f) for f in busquedas], ignore_index=True),
                             {"version": list(VERSIONES), "tarea": ["clf", "reg"], "modelo": ENSAMBLES})
    tabla_busqueda = tabla_busqueda[[c for c in tabla_busqueda if c != "p_objetivo_reg"] + ["p_objetivo_reg"]]
    tabla_busqueda.to_csv(f"{CARPETA_SALIDAS}P4b_busqueda.csv", index=False)
    MEJORES = mejores(tabla_busqueda, ESPACIOS)
else:   # algún cuaderno se corrió con EJECUTAR_BUSQUEDA = False: se usan los parámetros que guardó
    MEJORES = dict(sorted({k: v for m in ENSAMBLES for k, v in PARAMETROS[m]["mejores"].items()}.items(),
                          key=lambda kv: tuple(kv[0].split("|"))))
with open(f"{CARPETA_SALIDAS}P4b_mejores_parametros.json", "w") as f:
    json.dump({"espacios": ESPACIOS, "semilla": SEMILLA, "n_configs": N_CONFIGS_BUSQUEDA,
               "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
display(pd.DataFrame(MEJORES).T)

### 1.5 Búsqueda por función de pérdida (C3)

Mejor MAE de la búsqueda para cada función de pérdida de XGBoost y LightGBM (las configuraciones se repartieron por igual entre pérdidas).

In [ ]:
if "tabla_busqueda" in globals() and "p_objetivo_reg" in tabla_busqueda:
    por_perdida = (tabla_busqueda[(tabla_busqueda.tarea == "reg") & tabla_busqueda.p_objetivo_reg.notna()]
                   .assign(perdida=lambda t: t.p_objetivo_reg.map(PERDIDA_CORTA).fillna(t.p_objetivo_reg))
                   .groupby(["version", "modelo", "perdida"]).promedio.agg(configs="size", mejor_MAE_h="min", mediana_MAE_h="median")
                   .reset_index())
    display(por_perdida.round(4))
    por_perdida.to_csv(f"{CARPETA_SALIDAS}P4b_busqueda_por_perdida.csv", index=False)

## 2. Resultados de validación

### 2.1 Tabla general (los cinco bloques juntos)

In [ ]:
tabla = tabla_metricas(pred, ["version", "tarea", "modelo"])
display(tabla.round(4))
tabla.to_csv(f"{CARPETA_SALIDAS}P4b_tabla_validacion.csv", index=False)

### 2.2 Por bloque

Sirve para ver si el orden de los modelos se mantiene de un mes a otro.

In [ ]:
por_bloque = tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"])
por_bloque.to_csv(f"{CARPETA_SALIDAS}P4b_validacion_por_bloque.csv", index=False)
principal = por_bloque.assign(metrica=np.where(por_bloque.tarea == "clf", por_bloque.AP, por_bloque.MAE_h)) \
                      .pivot_table(index=["version", "tarea", "modelo"], columns="bloque", values="metrica")
print("Métrica principal por bloque (AP en clasificación, MAE en horas en regresión):")
display(principal.round(4))

### 2.3 Por segmento

Por categoría de zona, franja y, en regresión, decil de duración real (el decil 10 es el 10 % más lento).

In [ ]:
v = pred[pred.version == "internas"]
v = v.assign(decil_duracion=0)
v["decil_duracion"] = (pd.qcut(v.y.where(v.tarea == "reg"), 10, labels=False, duplicates="drop") + 1).fillna(0).astype(int)
segmentos = metricas_por_segmento(v, ["categoria_zona", "franja", "decil_duracion"])
segmentos = segmentos[~((segmentos.segmento == "decil_duracion") & (segmentos.tarea == "clf"))]
segmentos.to_csv(f"{CARPETA_SALIDAS}P4b_validacion_por_segmento.csv", index=False)
display(segmentos[segmentos.segmento != "decil_duracion"].round(3).head(40))

## 3. H1 y H2: diferencias con IC 95 % por bloques de día

Cada comparación se hace sobre **las mismas guías**. En AP, una diferencia **positiva** favorece al primero; en MAE, una **negativa**. Si el IC no incluye el 0, la diferencia es apreciable en el periodo evaluado.

- **H1:** cada ensamble frente al modelo lineal (logística en clasificación).
- **H1 con referencias del mismo tipo (C3, A5):** en regresión, cada pérdida de XGBoost y LightGBM frente a la referencia de su tipo (L1 contra la mediana zona×franja y el cuantil 0,5; L2 y Huber contra la media histórica y la lineal); en clasificación, cada ensamble frente a la tasa histórica y a «menor margen primero».
- **H2 (M3):** con clima frente a sin clima, para LightGBM y para la logística o lineal.

In [ ]:
filas = []
for e in ENSAMBLES:
    etiqueta = lambda t, e=e: f"H1 · {e} - {'Logística' if t == 'clf' else 'Lineal'}"
    filas += diferencias(pred, "internas", e, "internas", "Lineal", etiqueta, B_BOOTSTRAP, SEMILLA)

In [ ]:
# C3 y A5: frente a la referencia del mismo tipo
def agregar(tarea, va, ma, vb, mb, etiqueta):
    ic = diferencia_tarea(pred, tarea, va, ma, vb, mb, B_BOOTSTRAP, SEMILLA)
    if ic is not None:
        filas.append({"tarea": tarea, "metrica": "AP" if tarea == "clf" else "MAE_h", "comparacion": etiqueta, **ic})
for e in ENSAMBLES:
    for ref in ["Referencia_tasa_historica", "Referencia_menor_margen"]:
        agregar("clf", "internas", e, "internas", ref, f"H1 · {e} - {ref}")
for m in sorted(set(pred[pred.tarea == "reg"].modelo)):
    if "_" not in m or m.split("_")[0] not in ENSAMBLES: continue
    refs = ["Referencia_mediana", "Lineal_mediana"] if m.endswith("_L1") else ["Referencia", "Lineal"]
    for ref in refs:
        agregar("reg", "internas", m, "internas", ref, f"H1 · {m} - {ref}")

In [ ]:
for vclima in [x for x in VERSIONES if x.startswith("internas+clima")]:
    for mod in ["LightGBM", "Lineal"]:   # M3: también la logística o lineal
        etiqueta = lambda t, vc=vclima, mod=mod: f"H2 · {mod if mod != 'Lineal' else ('Logística' if t == 'clf' else 'Lineal')} {vc} - internas"
        filas += diferencias(pred, vclima, mod, "internas", mod, etiqueta, B_BOOTSTRAP, SEMILLA)

In [ ]:
diferencias_val = pd.DataFrame(filas)
diferencias_val["favorece_al_primero"] = np.where(diferencias_val.metrica == "AP",
                                                  diferencias_val.ic95_inf > 0, diferencias_val.ic95_sup < 0)
display(diferencias_val.round(4))
diferencias_val.to_csv(f"{CARPETA_SALIDAS}P4b_diferencias_validacion.csv", index=False)

## 4. Capacidad fija y modelo propuesto (C4)

Cada día se alertan las **k** guías de mayor riesgo (si el día tiene menos de k guías, todas). Así el número de alertas no depende de un umbral que pueda derivar entre periodos. Se reporta exhaustividad (incumplimientos detectados / incumplimientos) y precisión para k = 250, 500 y 1.000, en todos los modelos y referencias de clasificación.

In [ ]:
pclf = pred[(pred.version == "internas") & (pred.tarea == "clf")]
capacidad = pd.concat([tabla_capacidad(g.y.values, g.pred.values, g.dia.values, CAPACIDADES).assign(modelo=m)
                       for m, g in pclf.groupby("modelo", sort=False)], ignore_index=True)
capacidad = capacidad[["modelo"] + [x for x in capacidad if x != "modelo"]]
display(capacidad.round(3))
capacidad.to_csv(f"{CARPETA_SALIDAS}P4b_capacidad_validacion.csv", index=False)
cap_bloque = pd.concat([tabla_capacidad(g.y.values, g.pred.values, g.dia.values, [ALERTAS_POR_DIA]).assign(modelo=m, bloque=b)
                        for (m, b), g in pclf.groupby(["modelo", "bloque"], sort=False)], ignore_index=True)
cap_bloque.to_csv(f"{CARPETA_SALIDAS}P4b_capacidad_por_bloque.csv", index=False)
display(cap_bloque.pivot_table(index="modelo", columns="bloque", values="exhaustividad").round(3))

### 4.1 Modelo propuesto

- **Clasificación (C4):** el de mayor exhaustividad con k = 500 alertas por día entre la logística y los ensambles. Si una referencia lo supera, se avisa.
- **Regresión:** el de menor MAE con variables internas.

Aparte se anota el mejor ensamble de cada tarea (contraste de H1).

In [ ]:
k500 = capacidad[capacidad.k_por_dia == ALERTAS_POR_DIA].set_index("modelo")
ap = tabla[(tabla.version == "internas") & (tabla.tarea == "clf")].set_index("modelo").AP
cand_clf = k500.loc[[m for m in MODELOS if m in k500.index]].assign(AP=ap).sort_values(["exhaustividad", "AP"], ascending=False)
cand_reg = tabla[(tabla.version == "internas") & (tabla.tarea == "reg") & tabla.modelo.isin(MODELOS)].sort_values("MAE_h")
MODELO_CLF_PROPUESTO = cand_clf.index[0]
MODELO_REG_PROPUESTO = cand_reg.iloc[0].modelo
ENSAMBLE_H1 = {"clf": cand_clf[cand_clf.index.isin(ENSAMBLES)].index[0],
               "reg": cand_reg[cand_reg.modelo.isin(ENSAMBLES)].iloc[0].modelo}
display(cand_clf.round(4))
mejor_ref = k500.loc[[m for m in REFERENCIAS["clf"][1:] if m in k500.index]].exhaustividad   # la constante no ordena guías
if mejor_ref.max() >= cand_clf.exhaustividad.iloc[0]:
    print(f"AVISO: la referencia {mejor_ref.idxmax()} iguala o supera al modelo con k = {ALERTAS_POR_DIA} "
          f"({mejor_ref.max():.3f} frente a {cand_clf.exhaustividad.iloc[0]:.3f}). Discutir antes de registrar la decisión.")
print(f"Propuesto · clasificación: {MODELO_CLF_PROPUESTO} (exhaustividad k={ALERTAS_POR_DIA}: {cand_clf.exhaustividad.iloc[0]:.3f}) · "
      f"regresión: {MODELO_REG_PROPUESTO} (MAE {cand_reg.iloc[0].MAE_h:.3f} h)")
print(f"Mejor ensamble (H1) · clasificación: {ENSAMBLE_H1['clf']} · regresión: {ENSAMBLE_H1['reg']}")

In [ ]:
# Diferencia de exhaustividad con k = 500 frente a la logística y las referencias, IC 95 % por días
dif_cap = []
pa = pclf[pclf.modelo == MODELO_CLF_PROPUESTO].sort_values(["bloque", "guia"])
for otro in [m for m in ["Lineal"] + REFERENCIAS["clf"][1:] + ENSAMBLES if m != MODELO_CLF_PROPUESTO]:
    pb = pclf[pclf.modelo == otro].sort_values(["bloque", "guia"])
    if len(pb) == 0: continue
    assert (pa.guia.values == pb.guia.values).all()
    for k in CAPACIDADES:
        dif_cap.append({"comparacion": f"{MODELO_CLF_PROPUESTO} - {otro}", "k_por_dia": k,
                        **diferencia_capacidad_por_bloques(pa.y.values, pa.pred.values, pb.pred.values, pa.dia.values, k, B_BOOTSTRAP, SEMILLA)})
dif_cap = pd.DataFrame(dif_cap); display(dif_cap.round(4))
dif_cap.to_csv(f"{CARPETA_SALIDAS}P4b_capacidad_diferencias.csv", index=False)

## 5. Calibración y día de la semana

### 5.1 ¿Conviene calibrar?

Calibración isotónica evaluada sin mirar el bloque evaluado: se ajusta con los bloques anteriores y se aplica al siguiente. Se propone calibrar solo si mejora a la vez Brier y pérdida logarítmica. Con capacidad fija la calibración no cambia qué guías se alertan; sirve para leer la probabilidad.

In [ ]:
pc = pred[(pred.version == "internas") & (pred.tarea == "clf") & (pred.modelo == MODELO_CLF_PROPUESTO)]
cc = calibracion_cruzada(pc)
comp_cal = pd.DataFrame({
    "sin_calibrar": metricas("clf", cc.y, cc.pred),
    "isotonica_cruzada": metricas("clf", cc.y, cc.pred_cal)}).T[["n", "AP", "Brier", "LogLoss"]]
display(comp_cal.round(4))
comp_cal.to_csv(f"{CARPETA_SALIDAS}P4b_calibracion_comparacion.csv")
mejora_brier   = comp_cal.loc["isotonica_cruzada", "Brier"] < comp_cal.loc["sin_calibrar", "Brier"]
mejora_logloss = comp_cal.loc["isotonica_cruzada", "LogLoss"] < comp_cal.loc["sin_calibrar", "LogLoss"]
CALIBRACION_PROPUESTA = "isotonica" if (mejora_brier and mejora_logloss) else "ninguna"
print("Calibración propuesta:", CALIBRACION_PROPUESTA)

In [ ]:
calibrador = None
if CALIBRACION_PROPUESTA == "isotonica":
    calibrador = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(pc.pred, pc.y)
    joblib.dump(calibrador, f"{CARPETA_MODELOS}calibrador_clf_validacion.joblib")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for nombre, col in [("sin calibrar", "pred"), ("isotónica (cruzada)", "pred_cal")]:
    t = calibracion_tabla(cc.y.values, cc[col].values)
    ax.plot(t.p_media, t.y_obs, marker="o", label=nombre)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("Probabilidad predicha"); ax.set_ylabel("Frecuencia observada")
ax.set_title(f"Curva de calibración · {MODELO_CLF_PROPUESTO}"); ax.legend()
plt.tight_layout(); fig.savefig(f"{CARPETA_SALIDAS}P4b_curva_calibracion.png", dpi=130); plt.show()

### 5.2 Umbral equivalente (solo referencia)

La regla es la capacidad fija. Como referencia se muestra la tabla de umbrales y el umbral global que, en validación, daría en promedio k = 500 alertas por día; es el que derivó en el periodo julio–agosto anterior (868 alertas por día).

In [ ]:
p_val = calibrador.predict(pc.pred) if calibrador is not None else pc.pred.values
umbrales = tabla_umbrales(pc.y.values.astype(int), p_val, pc.dia.values)
umbrales["F1"] = 2 * umbrales.precision * umbrales.exhaustividad / (umbrales.precision + umbrales.exhaustividad).replace(0, np.nan)
display(umbrales.round(3))
umbrales.to_csv(f"{CARPETA_SALIDAS}P4b_tabla_umbrales.csv", index=False)

In [ ]:
UMBRAL_EQUIVALENTE = float(umbral_por_capacidad(p_val, pc.dia.values, ALERTAS_POR_DIA))
print(f"Umbral global equivalente a {ALERTAS_POR_DIA} alertas/día en validación: {UMBRAL_EQUIVALENTE:.4f} (solo referencia)")

### 5.3 Día de la semana (sábados)

Solo validación, modelo propuesto sin calibrar. Guías por día, prevalencia, probabilidad media y cuántas alertas tendría cada día con el umbral global equivalente frente a la capacidad fija.

In [ ]:
sab = pc.assign(dia_semana=pd.to_datetime(pc.dia, unit="D").dt.dayofweek, p=pc.pred.values,
               sobre_umbral=pc.pred.values >= UMBRAL_EQUIVALENTE,
               alerta_k=alertas_por_capacidad(pc.pred.values, pc.dia.values, ALERTAS_POR_DIA))
dias_sem = sab.groupby("dia_semana").dia.nunique()
tabla_dia = sab.groupby("dia_semana").agg(guias=("y", "size"), prevalencia=("y", "mean"), p_media=("p", "mean"),
                                          sobre_umbral=("sobre_umbral", "sum"), alertas_k=("alerta_k", "sum"),
                                          incumplimientos=("y", "sum"))
tabla_dia["dias"] = dias_sem
for col in ["guias", "sobre_umbral", "alertas_k", "incumplimientos"]:
    tabla_dia[f"{col}_por_dia"] = tabla_dia[col] / tabla_dia.dias
tabla_dia["pct_alertas_umbral"] = tabla_dia.sobre_umbral / tabla_dia.sobre_umbral.sum()
tabla_dia.index = tabla_dia.index.map(dict(enumerate(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])))
display(tabla_dia.round(3))
tabla_dia.to_csv(f"{CARPETA_SALIDAS}P4b_dia_semana_validacion.csv")

## 6. Decisión (antes de extraer o abrir septiembre y octubre)

La propuesta sale de las reglas fijadas: menor MAE en regresión, mayor exhaustividad con k = 500 en clasificación, calibración solo si mejora Brier y pérdida logarítmica, y alertas con capacidad fija. **El equipo puede cambiar cualquier valor**, pero solo con argumentos de validación.

Al poner `DECISION_APROBADA = True` y volver a ejecutar la celda, se escribe `P4b_decision_v2.json` con su fecha y su SHA-256, y ya no se sobrescribe. **Ese archivo se envía al director antes de extraer o abrir la reserva de septiembre y octubre** (C2).

In [ ]:
DECISION = {
    "version_decision": 2,
    "modelo_clf": MODELO_CLF_PROPUESTO, "modelo_reg": MODELO_REG_PROPUESTO, "version": "internas",
    "params_clf": params_de(MEJORES, PARAMS_BASE, "internas", "clf", MODELO_CLF_PROPUESTO),
    "params_reg": params_de(MEJORES, PARAMS_BASE, "internas", "reg", MODELO_REG_PROPUESTO),
    "ensamble_h1_clf": ENSAMBLE_H1["clf"], "ensamble_h1_reg": ENSAMBLE_H1["reg"],
    "params_h1_clf": params_de(MEJORES, PARAMS_BASE, "internas", "clf", ENSAMBLE_H1["clf"]),
    "params_h1_reg": params_de(MEJORES, PARAMS_BASE, "internas", "reg", ENSAMBLE_H1["reg"]),
    "calibracion": CALIBRACION_PROPUESTA,
    "regla_alertas": {"tipo": "capacidad_fija_por_dia", "k_por_dia": ALERTAS_POR_DIA, "sensibilidad_k": CAPACIDADES,
                      "umbral_global_equivalente_solo_referencia": UMBRAL_EQUIVALENTE},
    "referencias_regresion": ["Referencia (media historica zona x franja)", "Referencia_mediana", "Lineal", "Lineal_mediana"],
    "referencias_clasificacion": REFERENCIAS["clf"] + ["Lineal (logistica)"],
    "variables_categoricas": CATEGORICAS, "variables_numericas": VERSIONES["internas"],
    "excluir_entrega_sin_hora": EXCLUIR_ENTREGA_SIN_HORA, "usar_variables_cliente": USAR_VARIABLES_CLIENTE,
    "inicios_validacion": INICIOS_VALIDACION, "corte_observacion_desarrollo": CORTE_OBSERVACION,
    "fin_evaluable_desarrollo": FIN_EVALUABLE, "p99_t0_a_entrega_h": P99_T0_A_ENTREGA_H,
    "prueba": {"periodo": "T0 desde 2026-09-01 hasta el corte de la reserva menos p99_t0_a_entrega_h",
               "entrenamiento": "todas las guias con T0 y desenlace conocidos antes de 2026-09-01",
               "historia": "congelada en 2026-09-01", "ejecuciones_permitidas": 1},
    "huella_base_modelado_sha256": HUELLA_BASE, "semilla": SEMILLA}
# Si H2 resultó favorable en validación y el equipo decide usar clima, cambiar "version" y "variables_numericas" aquí.
print(json.dumps(DECISION, indent=2, default=str))

In [ ]:
DECISION_APROBADA = True     # poner True cuando el equipo esté de acuerdo, y volver a ejecutar esta celda
RUTA_DECISION = f"{CARPETA_SALIDAS}P4b_decision_v2.json"
if not DECISION_APROBADA:
    print("Decisión NO registrada (DECISION_APROBADA = False).")
elif os.path.exists(RUTA_DECISION):
    print("Ya existe una decisión registrada; no se sobrescribe:", RUTA_DECISION)
else:
    DECISION["fecha_registro"] = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=-5))).isoformat(timespec="seconds")
    with open(RUTA_DECISION, "w") as f: json.dump(DECISION, f, indent=2, default=str)
    print("Decisión registrada en", RUTA_DECISION)
if os.path.exists(RUTA_DECISION):
    print("Fecha de registro:", json.load(open(RUTA_DECISION))["fecha_registro"],
          "· SHA-256 del archivo:", hashlib.sha256(open(RUTA_DECISION, "rb").read()).hexdigest())

## 7. Manifiesto de la corrida

Versiones de librerías, huella de los archivos de entrada, parámetros, exclusiones y lista de salidas. Se ejecuta siempre al final.

In [ ]:
manifiesto = {"fecha_corrida": datetime.datetime.now().isoformat(), "python": sys.version.split()[0],
              "versiones": {"duckdb": duckdb.__version__, "pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
                            "lightgbm": lgb.__version__, "xgboost": xgb.__version__},
              "entradas": {"gold": huella(ruta), "calendario": huella(RUTA_CALENDARIO)},
              "huella_base_modelado_sha256": HUELLA_BASE,
              "parametros": {k: globals()[k] for k in ["INICIOS_VALIDACION", "FIN_DESARROLLO", "CORTE_OBSERVACION", "FIN_EVALUABLE",
                             "P99_T0_A_ENTREGA_H", "ALERTAS_POR_DIA", "CAPACIDADES", "SEMILLA", "USAR_VARIABLES_CLIENTE", "SOLO_SLA_UNICO",
                             "EXCLUIR_ENTREGA_SIN_HORA", "N_CONFIGS_BUSQUEDA", "MAX_FILAS_BUSQUEDA", "B_BOOTSTRAP"]},
              "exclusiones": exclusiones.to_dict("records"), "versiones_evaluadas": list(VERSIONES),
              "salidas": sorted(os.path.basename(f) for f in glob.glob(f"{CARPETA_SALIDAS}P4b_*"))}
with open(f"{CARPETA_SALIDAS}P4b_manifiesto.json", "w") as f: json.dump(manifiesto, f, indent=2, default=str)
print("Manifiesto guardado")

In [ ]:
import os, glob, zipfile, shutil
V4 = "/Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/"
archivos = [f for f in glob.glob(V4 + "**/*", recursive=True)
            if f.endswith((".csv", ".json", ".png")) and "/MODELADO/" not in f]
local = "/tmp/resultados_v4_0416.zip"
with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as z:
    for f in archivos:
        z.write(f, f.replace(V4, ""))
shutil.copy(local, V4 + "resultados_v4_0416.zip")
print(len(archivos), "archivos en", V4 + "resultados_v4_0416.zip")

## Qué entregar

De `REVISION_2026_11/V4/`: `P4b_exclusiones.csv` (de 0411), `P4b_busqueda.csv`, `P4b_busqueda_por_perdida.csv`, `P4b_mejores_parametros.json`, `P4b_tabla_validacion.csv`, `P4b_validacion_por_bloque.csv`, `P4b_validacion_por_segmento.csv`, `P4b_diferencias_validacion.csv`, `P4b_capacidad_validacion.csv`, `P4b_capacidad_por_bloque.csv`, `P4b_capacidad_diferencias.csv`, `P4b_calibracion_comparacion.csv`, `P4b_curva_calibracion.png`, `P4b_tabla_umbrales.csv`, `P4b_dia_semana_validacion.csv`, `P4b_manifiesto.json` y la decisión `P4b_decision_v2.json`.

**Cómo leer H1.** H1 se sostiene en validación si el IC 95 % de la diferencia «ensamble − lineal» (y frente a la referencia del mismo tipo) excluye el 0 en la dirección favorable. La confirmación es la prueba de septiembre y octubre, una sola vez, con la decisión ya registrada.